# Portfolio Dashboard

Notebook-first analytics for Schwab accounts.  
Edit the **Parameters** cell, then **Run All**.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from report import run_report
import plots
import pandas as pd

In [ ]:
# ============================================================
# PARAMETERS — edit this cell
# ============================================================

# Which accounts to include.
# Options: "Retirement", "Trade_and_leaps", or both.
ACCOUNTS = ["Retirement", "Trade_and_leaps"]

# Date range.  Both are required for meaningful annualized metrics.
# Use ISO format "YYYY-MM-DD".  Set to None only for exploratory runs.
START_DATE = "2024-04-15"
END_DATE   = None          # None = latest available date in the data

# Risk-free rate for Sharpe ratio (annualized decimal, e.g. 0.045 = 4.5%)
RISK_FREE_RATE = 0.0

# Set False to skip all benchmark work: no API call, no metrics column,
# no chart overlays, no benchmark artifacts in the report.
REPORT_INCLUDE_BENCHMARK = True
BENCHMARK = "SPY"          # ticker — only used when REPORT_INCLUDE_BENCHMARK = True

# Paths (relative to this notebook — all resolve into portfolio_dashboard/)
RAW_DIR       = "../data/raw"
PROCESSED_DIR = "../data/processed"
REPORTS_DIR   = "../reports"

In [ ]:
# ============================================================
# Load data and run pipeline
# ============================================================
results = run_report(
    raw_dir        = RAW_DIR,
    accounts       = ACCOUNTS,
    start_date     = START_DATE,
    end_date       = END_DATE,
    risk_free_rate = RISK_FREE_RATE,
    benchmark      = BENCHMARK if REPORT_INCLUDE_BENCHMARK else None,
)
print("Data loaded.  Date range:",
      results['nav'].index[0].date(), "\u2192",
      results['nav'].index[-1].date())

In [ ]:
# ============================================================
# Metrics summary
# ============================================================
results['metrics']

In [ ]:
# ============================================================
# Chart 1 — Portfolio value over time
# ============================================================
plots.plot_equity_curve(
    results['nav'],
    account_navs = results['account_navs'] if len(ACCOUNTS) > 1 else None,
    benchmark    = results['benchmark_returns'],
);

In [ ]:
# ============================================================
# Chart 2 — Cumulative investment return (TWR, $1 basis)
# ============================================================
plots.plot_cum_return(results['cum_return'], benchmark=results['benchmark_returns']);

In [ ]:
# ============================================================
# Chart 3 — Drawdown on TWR curve
# ============================================================
plots.plot_drawdown(results['drawdown']);

In [ ]:
# ============================================================
# Chart 4 — Portfolio NAV vs cumulative net cash flows
# ============================================================
plots.plot_nav_vs_cashflows(
    results['nav'],
    results['cumulative_cf'],
);

In [ ]:
# ============================================================
# Monthly returns table
# ============================================================
mon = results['monthly_rets'].dropna()
mon_pct = mon.map(lambda x: f"{x:.2%}")
mon_pct.name = "Monthly Return"
mon_pct

In [ ]:
# ============================================================
# Daily cash flows table (top 20 largest by absolute value)
# ============================================================
cf = results['daily_cf']
cf_nz = cf[cf != 0].sort_values(key=lambda s: s.abs(), ascending=False).head(20)
cf_nz.map(lambda x: f"${x:,.2f}").rename("Net External CF").to_frame()

In [ ]:
# ============================================================
# Generate final report
# ============================================================
from report import generate_final_report

report_path = generate_final_report(
    results,
    output_dir     = REPORTS_DIR,
    processed_base = PROCESSED_DIR,
)
print("Report saved:", report_path)